# Project 1: How Much Does NYC Pay Its Workers in 2025?   

NYC employs hundreds of thousands of people for the City to run.    
Aren't we curious to know what the city salary looks like in general?  
   
To find out, I dig into NYC's payroll records.

## Dataset Source 💻

I use dataset ["Citywide Payroll Data (Fiscal Year)"](https://data.cityofnewyork.us/City-Government/Citywide-Payroll-Data-Fiscal-Year-/k397-673e/data_preview) from NYC Open Data,  
provided by Office of Payroll Administration (OPA).

I filter the data in the website before downloading it:  
- Fiscal Year = 2025  
- Pay Basis = "per Annum"  
*(only employees with an annual salary, so that the salary unit is consistent).*  
  
The result is 330,289 rows.

### A Glimpse of Dead End 😵✨ 
  
In my first attempt, I used every row from the fiscal year of 2025 data,  
and the most common salary came out to $33.18.  
It turned out some workers are paid per day or per hour,  
so it was like mixing apples and oranges.  
I then went back to the source website and filtered only annual salaries.  

## The Easy Way: Pandas 🐼

Pandas is a Python tool that handles data tables for us,  
so we can calculate something, like an average, with a single command.  

1. First, I drop employee names for privacy.

In [22]:
import pandas as pd

df = pd.read_csv("Citywide_Payroll.csv", low_memory=False)

# drop name columns
df = df.drop(columns=["Last Name", "First Name", "Mid Init"])

df.head()

,Fiscal Year,Payroll Number,Agency Name,Agency Start Date,Work Location Borough,Title Description,Leave Status as of June 30,Base Salary,Pay Basis,Regular Hours,Regular Gross Paid,OT Hours,Total OT Paid,Total Other Pay
0,2025,67,ADMIN FOR CHILDREN'S SVCS,02/04/2013,MANHATTAN,PROGRAM EVALUATOR,ACTIVE,"$95,897.00",per Annum,"1,820","$89,097.13",0,$0.00,$699.41
1,2025,67,ADMIN FOR CHILDREN'S SVCS,02/22/2011,BRONX,CHILD PROTECTIVE SPECIALIST SUPERVISOR,ACTIVE,"$100,050.00",per Annum,"1,820","$96,959.10",73,"$4,291.75","$6,935.28"
2,2025,67,ADMIN FOR CHILDREN'S SVCS,02/22/2011,MANHATTAN,CHILD PROTECTIVE SPECIALIST SUPERVISOR,ACTIVE,"$106,346.00",per Annum,"1,820","$100,503.13",0,$0.00,$732.41
3,2025,67,ADMIN FOR CHILDREN'S SVCS,02/28/2011,MANHATTAN,AGENCY ATTORNEY,ACTIVE,"$127,802.00",per Annum,"1,820","$82,894.55",0,$0.00,"$9,804.97"
4,2025,67,ADMIN FOR CHILDREN'S SVCS,06/03/2024,BRONX,SPECIAL OFFICER,ACTIVE,"$39,206.00",per Annum,"2,080","$37,769.10",354.75,"$10,000.28","$7,737.53"


2. Then, I check the data type of "Base Salary".

In [23]:
df["Base Salary"].dtype

<StringDtype(storage='python', na_value=nan)>

3. Turns out, its data type is still read as *string* a.k.a text.   
Pandas read "Base Salary" as text because they are written with "\$" and ",", like "\$95,897.00".  
I need to remove the dollar signs and commas before I can do any calculation.

In [24]:
df["Base Salary"] = (
    df["Base Salary"].str.replace("$", "").str.replace(",", "").astype(float)
)

df["Base Salary"].dtype

dtype('float64')

4. Now, the data type of the "Base Salary" column is already converted to *float64*.  
This means that the values are now numeric and can be used for calculations.  
  
5. It's time to calculate **the mean, median, and mode** of the City's employees' salary!  
I am also curious who earns the mode salary the most, so let's find out. 

In [25]:
mean_salary = df["Base Salary"].mean()
median_salary = df["Base Salary"].median()
mode_salary = df["Base Salary"].mode()

print(f"Mean:   ${mean_salary:,.2f}")
print(f"Median: ${median_salary:,.2f}")
for a in mode_salary:
    print(f"Mode:   ${a:,.2f}")

most_common = df[df["Base Salary"].isin(mode_salary)]
most_common["Title Description"].value_counts()

Mean:   $91,827.35
Median: $90,375.00
Mode:   $109,352.00


Title Description
POLICE OFFICER    12317
FIREFIGHTER        6083
Name: count, dtype: int64

### What do these numbers mean?  
  
The mean (\$91,827) and median (\$90,375) are very close.  
This means there are not many very high salaries making the average higher.  
The mode, $109,352, is higher,  
and most of the workers who earn that amount are police officers and firefighters.

## The Hard Way 🏋🏻‍♀️  
  
Pandas did all the work! But let's try another way: **manually calculate them!**


### Step 1: Read the salaries from the file

I use Python's built-in "csv" module to read the file.    
Then I find the "Base Salary" column by its name in the header row,  
remove "$" and ",", change the data type, and collect all salaries into one list.  

In [26]:
import csv

salaries = []
with open("Citywide_Payroll.csv") as f:
    reader = csv.reader(f)
    # take the first row (column names)
    header = next(reader)
    col = header.index("Base Salary")
    for row in reader:
        salaries.append(float(row[col].replace("$", "").replace(",", "")))

### Step 2: Calculate the mean, median, and mode  

For **the mean**, I calculate it by adding up every salary,  
then divide by how many there are.  
  
For **the median**, I sort the salaries from lowest to highest and take the middle one.  
If there is an even number of salaries, there are two in the middle, so I average them.  
  
For **the mode**, I use a dictionary to count how many times each salary appears.  
Then I pick the salary (or salaries) with the highest count.  

In [27]:
# mean
mean_salary_hard = sum(salaries) / len(salaries)

# median
s = sorted(salaries)
n = len(s)
median_salary_hard = (s[(n - 1) // 2] + s[n // 2]) / 2

# mode
counts = {}
for x in salaries:
    counts[x] = counts.get(x, 0) + 1
highest = max(counts.values())
modes = [x for x in counts if counts[x] == highest]

print(f"Mean:   ${mean_salary_hard:,.2f}")
print(f"Median: ${median_salary_hard:,.2f}")
for b in modes:
    print(f"Mode:   ${b:,.2f} (appears {highest:,} times)")

Mean:   $91,827.35
Median: $90,375.00
Mode:   $109,352.00 (appears 18,400 times)


### Step 3: Are the numbers from the hard way similar to the easy way?  

**Yes, they are! 👏🏻** The hard way gives exactly the same results as Pandas!

## Visualization 📊 

I group salaries into $50k ranges,  
and create the histogram to visualize the data.   

In [28]:
# width of each bin
bin_size = 50_000
# maximum length of the bar
max_bar = 50

# calculate the number of employees in each bin
bins = {}
for salary in salaries:
    start = int(salary // bin_size) * bin_size
    bins[start] = bins.get(start, 0) + 1

biggest = max(bins.values())


print("Annual Salary Distribution for New York City Employees (FY2025)")

for start in range(0, max(bins) + bin_size, bin_size):
    count = bins.get(start, 0)
    bar = "█" * round(count / biggest * max_bar)
    if count > 0 and bar == "":
        bar = "▏"
    label = f"${start // 1000:>3}k–${(start + bin_size) // 1000:>3}k"
    print(f"{label} | {bar} {count:,}")

Annual Salary Distribution for New York City Employees (FY2025)
$  0k–$ 50k | █████████████ 40,116
$ 50k–$100k | ██████████████████████████████████████████████████ 149,534
$100k–$150k | █████████████████████████████████████████ 123,347
$150k–$200k | ████ 13,077
$200k–$250k | █ 3,630
$250k–$300k | ▏ 569
$300k–$350k | ▏ 11
$350k–$400k | ▏ 3
$400k–$450k | ▏ 2


The chart shows that **the peak is around \$50k–\$100k**,  
and only a few earn above $250k.

## Conclusion 📝 
  
A typical salaried NYC employee earns about \$90k and \$92k a year  
(the median is \$90,375 and the mean is \$91,827),  
with 18,400 people earn exactly $109,352 (the mode).  
Most employees (over 270,000 of them) earn between \$50k and \$150k a year,  
with the peak is in \$100k-\$150k range, while very few earn more than $250k.  

*Limitations: this project only covers annual-salary employees and only base pay (no overtime).*  
